# Option A - OCR-grounded reconstruction (fast check)

**The recommended first thing to try, and the cheapest.** No training, no new model - it reuses the VLM you already serve.

Prompt-only `pixels -> HTML` asks one model to *read*, *localise*, and *emit nested HTML* all at once, and the errors compound - which is why even the 35B teacher garbles the tables. Here an OCR engine does the reading and localising, and the VLM's job collapses to **just the structure**: which cells merge, which rows are headers.

Everything runs on `localhost` - the invoices never leave the network.

**What this checks:** on one invoice, does handing the model the OCR text+positions fix the reconstruction where prompt-only fails? If yes, the same grounding fixes the poisoned teacher labels.

## Config
One cell for everything you'd change.

In [ ]:
import sys, base64, mimetypes
from pathlib import Path
from openai import OpenAI   # pip install openai (talks to the vLLM OpenAI-compatible server)

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# --- vLLM endpoint: the SAME served model you already run -------------
BASE_URL   = 'http://localhost:8000/v1'    # OpenAI-compatible vLLM server
MODEL_NAME = 'qwen3.6-35b-a3b-fp8'          # the 35B teacher (or any served VLM)
API_KEY    = 'EMPTY'                         # vLLM ignores it; must be non-empty

# --- data (confidential; stays on the network) -----------------------
IMAGES_DIR = ROOT / 'data' / 'invoices'
IMAGE_GLOB = ('*.png', '*.jpg', '*.jpeg', '*.webp', '*.tif', '*.tiff')

# --- OCR-layout knobs (expect to tune these on real invoices) --------
OCR_STYLE = 'grid'   # 'grid' = aligned text table, 'coords' = text @ (x,y)
ROW_TOL   = 0.6      # row-clustering tolerance, in median line-heights
COL_GAP   = 1.0      # min whitespace corridor width to call a column break

# --- decoding --------------------------------------------------------
TEMPERATURE, MAX_TOKENS, REQUEST_TIMEOUT = 0.0, 4096, 300

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)   # localhost only
images = sorted(p for pat in IMAGE_GLOB for p in IMAGES_DIR.glob(pat))
print(len(images), 'invoices in', IMAGES_DIR)


## OCR engine
This needs an on-prem OCR engine. Default is **PaddleOCR** (offline, returns text + boxes). Install once on the serving box:
```bash
pip install paddleocr paddlepaddle
```
The engine is pluggable in [src/ocr/engine.py](src/ocr/engine.py) - swap PaddleOCR for Surya or your in-house OCR in that one file and nothing below changes. If your PaddleOCR version returns a different result shape, adjust `PaddleOcrEngine.read` there (that's the only place that touches the engine API).

In [ ]:
from src.ocr.engine import run_ocr
from src.ocr.layout import serialize_layout, to_grid_html
from src.model.prompts import GROUNDED_INSTRUCTION, format_layout_block, clean_prediction
from src.data.html_utils import extract_cells

# Today's failing path: prompt-only, no OCR. Kept as the baseline to beat --
# this is the teacher notebook's direct instruction, trimmed.
PROMPT_ONLY_INSTRUCTION = (
    'Reconstruct the printed data table in this document as HTML. Preserve every '
    'rowspan, colspan, merged cell, and header hierarchy (use <th> for headers), '
    'keep the reading order, and transcribe each cell. Ignore handwriting, stamps, '
    'logos, QR codes and barcodes. Output raw HTML starting with <table> and ending '
    'with </table>. No markdown fences, no explanation.'
)


def data_url(path):
    mime = mimetypes.guess_type(str(path))[0] or 'image/png'
    return f'data:{mime};base64,' + base64.b64encode(Path(path).read_bytes()).decode()


def ask(image_path, instruction, ocr_layout=None):
    """One call to the served VLM via the OpenAI SDK. When ocr_layout is given it
    is appended to the instruction *exactly* as src.model.prompts.build_messages
    does -- so this notebook stays prompt-identical to the training path (the
    invariant that a train/inference prompt mismatch throws the adapter away)."""
    text = instruction if ocr_layout is None else f'{instruction}\n\n{format_layout_block(ocr_layout)}'
    resp = client.chat.completions.create(
        model=MODEL_NAME, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
        timeout=REQUEST_TIMEOUT,
        messages=[{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': data_url(image_path)}},
            {'type': 'text', 'text': text},
        ]}],
    )
    return clean_prediction(resp.choices[0].message.content)


def summary(tag, html):
    cells = extract_cells(html)
    spans = sum(c.is_spanning for c in cells)
    print(f'{tag:12s} cells={len(cells):3d} spanning={spans:3d} '
          f'starts_with_table={html.startswith("<table")}')


## Step 1 - look at the OCR layout *before* the VLM
The cheap diagnostic. If the deterministic grid is already wrong (a column split through a cell, two rows merged), fix it here by tuning `ROW_TOL` / `COL_GAP` - **no GPU needed**. A VLM can't rescue a grid that's garbled going in.

In [ ]:
from IPython.display import HTML, Image as IPyImage, display

assert images, f'no invoices in {IMAGES_DIR}'
img = images[0]

words = run_ocr(img)
print(len(words), 'OCR words')

layout = serialize_layout(words, style=OCR_STYLE, row_tol=ROW_TOL, col_gap=COL_GAP)
print('\n--- serialized layout (this is what the VLM gets handed) ---\n')
print(layout[:2000])

# Zero-model floor: rectangular HTML from geometry alone, no merged cells.
floor = to_grid_html(words, row_tol=ROW_TOL, col_gap=COL_GAP)
print('\n--- image | zero-model floor ---')
display(IPyImage(filename=str(img), width=420))
display(HTML(floor or '<i>empty grid</i>'))


## Step 2 - the fast check: prompt-only vs grounded vs floor
Same invoice, three reconstructions:
- **prompt-only** - today's failing path;
- **OCR-grounded** - Option A;
- **floor** - the zero-model rectangular grid (no merged cells).

If grounded clearly beats prompt-only on the merged cells and header rows, the hypothesis holds. If grounded barely beats the *floor*, the VLM isn't adding structure and the problem is upstream (OCR or thresholds).

In [ ]:
baseline = ask(img, PROMPT_ONLY_INSTRUCTION)                    # today's approach
grounded = ask(img, GROUNDED_INSTRUCTION, ocr_layout=layout)    # Option A

summary('prompt-only', baseline)
summary('grounded', grounded)
summary('floor', floor)

print('\n=== image ==='); display(IPyImage(filename=str(img), width=420))
print('=== prompt-only (baseline) ==='); display(HTML(baseline or '<i>empty</i>'))
print('=== OCR-grounded (Option A) ==='); display(HTML(grounded or '<i>empty</i>'))


## Step 3 (optional) - batch grounded labels
If grounding wins the eyeball test, re-label every invoice with grounding and write a manifest that drops straight into [finetune-and-serve.ipynb](finetune-and-serve.ipynb) in place of the poisoned teacher labels. Idempotent - a server hiccup mid-batch doesn't cost earned labels.

In [ ]:
import json

OUT_DIR  = ROOT / 'data' / 'teacher-grounded'
OUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST = OUT_DIR / 'labels.jsonl'   # drops into finetune-and-serve.ipynb as-is

done = set()
if MANIFEST.exists():
    done = {json.loads(l)['uid'] for l in MANIFEST.read_text().splitlines() if l.strip()}
print(len(done), 'already done;', len(images) - len(done), 'to go')

with MANIFEST.open('a') as mf:
    for i, path in enumerate(images, 1):
        uid = path.stem
        if uid in done:
            continue
        try:
            words = run_ocr(path)
            layout = serialize_layout(words, style=OCR_STYLE, row_tol=ROW_TOL, col_gap=COL_GAP)
            html = ask(path, GROUNDED_INSTRUCTION, ocr_layout=layout)
        except Exception as e:
            print(f'  [{i}/{len(images)}] {uid}: FAILED {e}')
            continue
        if not html.startswith('<table'):
            print(f'  [{i}/{len(images)}] {uid}: no table parsed -- skipped')
            continue
        rec = {'uid': uid, 'image_path': str(path), 'html': html,
               'model': MODEL_NAME, 'grounded': True, 'ocr_style': OCR_STYLE}
        mf.write(json.dumps(rec, ensure_ascii=False) + '\n'); mf.flush()
        print(f'  [{i}/{len(images)}] {uid}: ok ({len(html)} chars)')
print('grounded labels ->', MANIFEST)


---
**Read the result honestly.** With no ground truth your eyes are still the eval set - human-review these labels before they train anything. If grounded beats prompt-only, tell me and I'll wire this into [teacher-label-tables.ipynb](teacher-label-tables.ipynb) and [finetune-and-serve.ipynb](finetune-and-serve.ipynb) properly, then build the 20 invoices into a TEDS-scored eval set so "does grounding help?" becomes a number with a confidence interval instead of an impression.